# 05 - The RAG answer chain

**Goal:** turn retrieved passages into a short answer where every claim carries a citation - or into an honest "not covered".

The steps for one question:

1. Detect the question's language (German or English; anything else is answered in English).
2. Retrieve passages (notebook 04).
3. Build a **strict prompt**: answer only from the numbered passages, cite with `[n]`, say `NOT_COVERED` if the passages don't answer.
4. Call the LLM (temperature 0).
5. **Parse** the output: keep valid `[n]` markers, turn them into citations (label + EUR-Lex link). No valid citation at all -> treat it as a refusal.

**LLM note:** if `GOOGLE_API_KEY` is set in `.env`, this notebook calls real Gemini. Otherwise it uses a clearly labelled *fake* LLM with canned answers so you can still see every step.

In [1]:
import os
import warnings

# Silence noise before anything is imported (the sunset notice, progress bars, HF logs).
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_VERBOSITY"] = "error"

from pathlib import Path  # noqa: E402

from langchain_core.messages import AIMessage  # noqa: E402
from langchain_core.runnables import RunnableLambda  # noqa: E402

from assistant.chain import (  # noqa: E402
    NOT_COVERED_TOKEN,
    PROMPT,
    answer_question,
    detect_language,
    format_passages,
    make_llm,
)
from assistant.config import get_settings  # noqa: E402
from assistant.index import load_index, make_embeddings  # noqa: E402
from assistant.parse import read_chunks  # noqa: E402
from assistant.retrieve import build_retriever  # noqa: E402

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # run from the project root so data/ paths work

settings = get_settings()
chunks = read_chunks(settings.chunks_path)
store = load_index(make_embeddings(settings), settings.chroma_dir)
retrieve = build_retriever(store, chunks, settings)

if settings.google_api_key or settings.openai_api_key:
    llm = make_llm(settings)
    print(f"Using the real LLM ({settings.llm_provider}).")
else:

    def fake_llm(prompt_value):
        """FAKE model: no AI involved. Refuses 'France' questions, otherwise cites [1] and [2]."""
        text = prompt_value.to_string()
        question = text.rsplit("Question:", 1)[-1]
        if "France" in question:
            out = NOT_COVERED_TOKEN
        elif "Antwort in German" in text or "in German" in text:
            out = "Das ergibt sich aus den bereitgestellten Textstellen [1]. Ergaenzend siehe [2]."
        else:
            out = "This follows from the provided passages [1]. See also [2][9]."
        usage = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}
        return AIMessage(content=out, usage_metadata=usage)

    llm = RunnableLambda(fake_llm)
    print("NOTE: no API key found -> using a FAKE LLM with canned answers (not real AI).")
    print("Add GOOGLE_API_KEY to .env and re-run for real answers.")

W1123 20:02:51.022000 12595 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


NOTE: no API key found -> using a FAKE LLM with canned answers (not real AI).
Add GOOGLE_API_KEY to .env and re-run for real answers.


## The prompt template

This is the whole "instruction manual" for the model. `{language}`, `{token}`, `{passages}` and `{question}` are filled in per question.

In [2]:
system, human = (m.prompt.template for m in PROMPT.messages)
for role, msg in [("SYSTEM", system), ("HUMAN", human)]:
    print(f"--- {role} ---\n{msg}\n")

--- SYSTEM ---
You are an assistant that answers questions about the EU AI Act and the GDPR.

Rules:
1. Answer ONLY from the numbered passages below. Do not use any other knowledge.
2. Write the answer in {language}.
3. Cite every claim with the number of the passage that supports it, like [1] or [2][3].
   Use only the numbers of the passages given. Never invent a number.
4. If the passages do not contain the answer, reply with exactly {token} and nothing else.
5. Be concise and precise. Do not give legal advice.

--- HUMAN ---
Passages:
{passages}

Question: {question}



## Step by step for one question

We run the steps by hand for a German question, so you can see what the model receives and returns.

In [3]:
question = "Welche KI-Praktiken sind nach der KI-Verordnung verboten?"
language = detect_language(question)
docs = retrieve(question, language)
print("language:", language, "| passages retrieved:", len(docs))
for i, d in enumerate(docs, 1):
    print(f"  [{i}] {d.page_content[:90]!r}")

language: de | passages retrieved: 6
  [1] 'Artikel 5 – Verbotene Praktiken im KI-Bereich\n(1) Folgende Praktiken im KI-Bereich sind ve'
  [2] 'Artikel 5 – Verbotene Praktiken im KI-Bereich\nb) Die Verwendung eines KI-Systems, mit dem '
  [3] 'Erwägungsgrund 28\nAbgesehen von den zahlreichen nutzbringenden Verwendungsmöglichkeiten vo'
  [4] 'Artikel 5 – Verbotene Praktiken im KI-Bereich\n(8) Dieser Artikel berührt nicht die Verbote'
  [5] 'Artikel 5 – Verbotene Praktiken im KI-Bereich\n(4) Unbeschadet des Absatzes 3 wird jede Ver'
  [6] 'Artikel 1 – Gegenstand\n(2) In dieser Verordnung wird Folgendes festgelegt:\na) harmonisiert'


In [4]:
prompt_value = PROMPT.invoke(
    {
        "language": "German" if language == "de" else "English",
        "token": NOT_COVERED_TOKEN,
        "passages": format_passages(docs),
        "question": question,
    }
)
text = prompt_value.to_string()
print(f"Prompt sent to the model: {len(text)} characters. Start and end:\n")
print(text[:700], "\n[...]\n", text[-200:])

Prompt sent to the model: 4066 characters. Start and end:

System: You are an assistant that answers questions about the EU AI Act and the GDPR.

Rules:
1. Answer ONLY from the numbered passages below. Do not use any other knowledge.
2. Write the answer in German.
3. Cite every claim with the number of the passage that supports it, like [1] or [2][3].
   Use only the numbers of the passages given. Never invent a number.
4. If the passages do not contain the answer, reply with exactly NOT_COVERED and nothing else.
5. Be concise and precise. Do not give legal advice.
Human: Passages:
[1] AI Act Art. 5(1) (de): Artikel 5 – Verbotene Praktiken im KI-Bereich
(1) Folgende Praktiken im KI-Bereich sind verboten:
a) das Inverkehrbringen, die Inbetriebnahme o 
[...]
 sonderem Augenmerk auf kleinen Midcap-Unternehmen und kleinen und mittleren Unternehmen (KMU), einschließlich Start-up-Unternehmen.

Question: Welche KI-Praktiken sind nach der KI-Verordnung verboten?


In [5]:
raw = (PROMPT | llm).invoke(
    {
        "language": "German",
        "token": NOT_COVERED_TOKEN,
        "passages": format_passages(docs),
        "question": question,
    }
)
print("RAW model output:\n", raw.content)

RAW model output:
 Das ergibt sich aus den bereitgestellten Textstellen [1]. Ergaenzend siehe [2].


## The full chain on three questions

`answer_question` does everything above plus parsing. Watch the third one: an out-of-scope question must be **refused**, not answered from the model's general knowledge.

In [6]:
def show(question):
    ans = answer_question(question, retrieve, llm)
    print("Q:", question)
    tokens = f"{ans.input_tokens}/{ans.output_tokens}"
    print(f"language={ans.language}  refused={ans.refused}  {ans.latency_ms} ms  tokens={tokens}")
    print("A:", ans.answer)
    for c in ans.citations:
        print(f"   - {c.label} ({c.language})  {c.source_url}")
    print()


show("Welche KI-Praktiken sind nach der KI-Verordnung verboten?")
show("When is personal data considered anonymous under the GDPR?")
show("What is the capital of France?")

Q: Welche KI-Praktiken sind nach der KI-Verordnung verboten?
language=de  refused=False  25 ms  tokens=0/0
A: Das ergibt sich aus den bereitgestellten Textstellen [1]. Ergaenzend siehe [2].
   - AI Act Art. 5(1) (de)  https://eur-lex.europa.eu/legal-content/DE/TXT/?uri=CELEX:02024R1689-20260727#art_5
   - AI Act Art. 5(1a) (de)  https://eur-lex.europa.eu/legal-content/DE/TXT/?uri=CELEX:02024R1689-20260727#art_5

Q: When is personal data considered anonymous under the GDPR?
language=en  refused=False  29 ms  tokens=0/0
A: This follows from the provided passages [1]. See also [2].
   - GDPR Recital 26 (en)  https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32016R0679#rct_26
   - AI Act Recital 54 (en)  https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689#rct_54

Q: What is the capital of France?
language=en  refused=True  30 ms  tokens=0/0
A: This question is not covered by the text of the EU AI Act or the GDPR.



## What to notice

- The prompt carries the *rules*; the passages carry the *facts*. The model is not asked what it knows.
- Citation markers `[n]` are only a pointer into the numbered passage list. `parse_citations` turns them into labels and EUR-Lex links, and silently drops numbers that do not exist (the fake model deliberately emits `[9]` once to show this).
- A refusal can happen two ways: the model says `NOT_COVERED`, or its answer has no valid citation at all.